In [2]:
from pathlib import Path
import re
import json


# ============================================================
# Configuration
# ============================================================

KNOWLEDGE_BASE = Path(r"C:\Users\User\Nasa Rag SYS\knowledge_base\experiments")

OUTPUT_DIR = Path("processed/chunks")
OUTPUT_FILE = OUTPUT_DIR / "chunks.json"


# ============================================================
# Parse YAML-like front matter
# ============================================================

def parse_metadata(content):

    metadata = {}

    match = re.match(
        r"^---\s*\n(.*?)\n---\s*\n",
        content,
        re.DOTALL
    )

    if not match:
        return metadata, content

    metadata_text = match.group(1)

    for line in metadata_text.splitlines():

        if ":" not in line:
            continue

        key, value = line.split(":", 1)

        key = key.strip()
        value = value.strip()

        # Convert numeric values
        try:
            if "." in value:
                value = float(value)
            else:
                value = int(value)
        except ValueError:
            pass

        metadata[key] = value

    document_text = content[match.end():]

    return metadata, document_text.strip()


# ============================================================
# Extract sections
# ============================================================

def extract_sections(document_text):

    sections = {}

    pattern = r"^##\s+(.+?)\s*$"

    matches = list(
        re.finditer(
            pattern,
            document_text,
            re.MULTILINE
        )
    )

    for i, match in enumerate(matches):

        section_name = match.group(1).strip()

        start = match.end()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(document_text)

        section_content = document_text[start:end].strip()

        sections[section_name] = section_content

    return sections


# ============================================================
# Create chunks for one experiment
# ============================================================

def create_chunks(content):

    metadata, document_text = parse_metadata(content)

    sections = extract_sections(document_text)

    experiment_id = metadata.get(
        "experiment_id",
        "unknown"
    )

    chunks = []

    # --------------------------------------------------------
    # Chunk 1: Investigation + Material
    # --------------------------------------------------------

    investigation = sections.get(
        "Investigation",
        ""
    )

    material = sections.get(
        "Material",
        ""
    )

    if investigation or material:

        text = f"""# Experiment {experiment_id}

## Investigation

{investigation}

## Material

{material}
"""

        chunks.append({
            "text": text.strip(),
            "metadata": {
                **metadata,
                "chunk_type": "investigation_material"
            }
        })

    # --------------------------------------------------------
    # Chunk 2: Experimental Conditions
    # --------------------------------------------------------

    conditions = sections.get(
        "Experimental Conditions",
        ""
    )

    if conditions:

        text = f"""# Experiment {experiment_id}

## Experimental Conditions

{conditions}
"""

        chunks.append({
            "text": text.strip(),
            "metadata": {
                **metadata,
                "chunk_type": "experimental_conditions"
            }
        })

    # --------------------------------------------------------
    # Chunk 3: Observed Results
    # --------------------------------------------------------

    results = sections.get(
        "Observed Results",
        ""
    )

    if results:

        text = f"""# Experiment {experiment_id}

## Observed Results

{results}
"""

        chunks.append({
            "text": text.strip(),
            "metadata": {
                **metadata,
                "chunk_type": "observed_results"
            }
        })

    # --------------------------------------------------------
    # Chunk 4: Fire Risk + Summary
    # --------------------------------------------------------

    fire_risk = sections.get(
        "Fire Risk",
        ""
    )

    summary = sections.get(
        "Experimental Summary",
        ""
    )

    if fire_risk or summary:

        text = f"""# Experiment {experiment_id}

## Fire Risk

{fire_risk}

## Experimental Summary

{summary}
"""

        chunks.append({
            "text": text.strip(),
            "metadata": {
                **metadata,
                "chunk_type": "risk_summary"
            }
        })

    return chunks


# ============================================================
# Load all experiments and create chunks
# ============================================================

def chunk_knowledge_base():

    all_chunks = []

    files = sorted(
        KNOWLEDGE_BASE.glob("*.md")
    )

    print(f"Found {len(files)} Markdown files.")

    for file_path in files:

        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as f:

            content = f.read()

        chunks = create_chunks(content)

        for chunk in chunks:

            chunk["metadata"]["source"] = (
                file_path.name
            )

            all_chunks.append(chunk)

    return all_chunks


# ============================================================
# Save chunks
# ============================================================

def save_chunks(chunks):

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    # Add stable chunk IDs
    for index, chunk in enumerate(chunks):

        chunk["chunk_id"] = index

    with open(
        OUTPUT_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            chunks,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(f"\nChunks saved to:")
    print(OUTPUT_FILE.resolve())


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    chunks = chunk_knowledge_base()

    print("\n================================")
    print("CHUNKING COMPLETE")
    print("================================")

    print(f"Total chunks: {len(chunks)}")

    save_chunks(chunks)

    print("\nFirst chunk:")
    print("--------------------------------")

    print(chunks[0]["text"])

    print("\nMetadata:")
    print("--------------------------------")

    print(chunks[0]["metadata"])

    print("\nChunk ID:")
    print("--------------------------------")

    print(chunks[0]["chunk_id"])

Found 300 Markdown files.

CHUNKING COMPLETE
Total chunks: 1200

Chunks saved to:
C:\Users\User\Nasa Rag SYS\rag\ingestion\processed\chunks\chunks.json

First chunk:
--------------------------------
# Experiment EXP-0001

## Investigation

- **Source Investigation:** Saffire-III

## Material

- **Material:** Silicone rubber
- **Material Type:** solid

Metadata:
--------------------------------
{'experiment_id': 'EXP-0001', 'source_investigation': 'Saffire-III', 'material': 'Silicone rubber', 'material_type': 'solid', 'flow_configuration': 'concurrent', 'oxygen_concentration_pct': 14.47, 'pressure_kpa': 90.0, 'flow_velocity_cm_s': 23.87, 'ambient_temperature_c': 22.5, 'outcome': 'extinguished', 'risk_level': 'medium', 'test_date': '2026-07-15', 'chunk_type': 'investigation_material', 'source': 'EXP-0001.md'}

Chunk ID:
--------------------------------
0
